In [32]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Administrator\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Administrator\AppData\Local\Programs\Python\Python310\python.exe"

# If you're in a notebook, it's common to have an old SparkSession running.
# This makes the cell re-runnable safely.
spark = SparkSession.builder.master("local[2]").appName("pyspark-local").getOrCreate()
sc = spark.sparkContext

sc.setLogLevel("ERROR")

#Task 1: Load CSV Data
customer_df = spark.read.csv("sample_data.csv", header=True, inferSchema=True)
#customer_df.printSchema()
#customer_df.show(5)

customer_df_dervied = customer_df.withColumn("total_amount", customer_df["quantity"] * customer_df["price"])
customer_df_dervied_filtered = customer_df_dervied.filter((customer_df_dervied["category"] == "Electronics") & (customer_df_dervied["total_amount"] > 20000))
customer_df_dervied_filtered.show()

#task4  : Total revenue by category
customer_df_dervied_tot_rev =  customer_df_dervied.groupBy("category").sum("total_amount").withColumnRenamed("sum(total_amount)", "total_revenue")
#customer_df_dervied_tot_rev.sort("total_revenue", ascending=False).show()

#Task 5: Total orders by city
customer_df_dervied_tot_count = customer_df_dervied.groupBy("city").agg(count("order_id")).withColumnRenamed("count(order_id)", "order_count").show()

#Task 6: Total revenue by payment mode
customer_df_dervied_tot_pymt_mode = customer_df_dervied.groupBy("payment_mode").agg(sum("total_amount")).withColumnRenamed("sum(total_amount)", "total_rev_per_payment_mode")
#customer_df_dervied_tot_pymt_mode.filter(col("total_rev_per_payment_mode") > 30000).show()

#Task 7: Top customer by total amount spent
customer_df_dervied.sort("total_amount", ascending=False).show(1)

#Task 8 
customer_df_dervied_tot_rev.write.mode("overwrite").option("header", "true").csv("sample_data_result.csv")


+--------+----------+-----------+-------+-----------+--------+-----+------------+---------+------------+
|order_id|order_date|customer_id|product|   category|quantity|price|payment_mode|     city|total_amount|
+--------+----------+-----------+-------+-----------+--------+-----+------------+---------+------------+
|     101|2024-01-01|       C001| Laptop|Electronics|       1|70000| Credit Card|Bangalore|       70000|
|     102|2024-01-01|       C002| Mobile|Electronics|       2|20000|         UPI|  Chennai|       40000|
|     107|2024-01-04|       C005| Tablet|Electronics|       1|25000| Credit Card|    Delhi|       25000|
|     108|2024-01-04|       C003| Laptop|Electronics|       1|68000|  Debit Card|Hyderabad|       68000|
+--------+----------+-----------+-------+-----------+--------+-----+------------+---------+------------+

+---------+-----------+
|     city|order_count|
+---------+-----------+
|Bangalore|          2|
|  Chennai|          2|
|     Pune|          1|
|    Delhi|    

Py4JJavaError: An error occurred while calling o959.csv.
: ExitCodeException exitCode=-1073741515: 
	at org.apache.hadoop.util.Shell.runCommand(Shell.java:1007)
	at org.apache.hadoop.util.Shell.run(Shell.java:900)
	at org.apache.hadoop.util.Shell$ShellCommandExecutor.execute(Shell.java:1212)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1306)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1288)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:978)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:660)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:700)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:672)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:699)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:672)
	at org.apache.hadoop.fs.RawLo